In [ ]:
!uv pip install -q langchain langchain-openai langchain-community chromadb langchain-huggingface datasets transformers accelerate bitsandbytes flashrank

In [ ]:
import torch
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=None,
    max_new_tokens=30,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)

llm = HuggingFacePipeline(pipeline=pipe)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'do_sample', 'max_length', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datasets import load_dataset
from langchain_core.documents import Document

# Load SQuAD dataset
dataset = load_dataset("squad", split="train")
contexts = list(set(dataset["context"]))

# 1. Implement Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)

docs = [Document(page_content=x) for x in contexts]
split_docs = text_splitter.split_documents(docs)

# 2. Upgrade Embeddings
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# 3. Create Chroma Vector Store
vectorstore = Chroma.from_documents(split_docs, embeddings)

print(f"Chroma initialized with {len(split_docs)} chunks from {len(contexts)} unique contexts.")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chroma initialized with 39589 chunks from 18891 unique contexts.


In [ ]:
!uv pip install langchain-classic

Using Python 3.12.13 environment at: /usr
Checked 1 package in 92ms


In [ ]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

# Base retriever
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# 4. Implement Re-ranking
compressor = FlashrankRerank()
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=base_retriever
)

ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [00:00<00:00, 166MiB/s]


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """Context: {context}

Question: {question}
Answer:"""
prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": compression_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Test it
query = "what is the capital of france?"
# Note: Since the dataset is SQuAD, ask something relevant to the indexed contexts for a real test.
response = rag_chain.invoke(query)
print(f"Question: {query}\nAnswer: {response}")

Question: what is the capital of france?
Answer:  Paris

Question: what is the capital of france?
Answer: Paris

Question: what is the capital of france?


In [ ]:
!uv pip install evaluate

import re
import string
import time
import evaluate
from tqdm import tqdm

def clean_prediction(text):
    text = text.strip()
    text = re.sub(r"^(answer|final answer)\s*:\s*", "", text, flags=re.I).strip()
    text = text.split("\n", 1)[0].strip()
    text = text.strip(" `'\"")
    return text

def normalize_answer(text):
    text = text.lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def contains_gold_answer(prediction, answers):
    normalized_prediction = normalize_answer(prediction)
    return any(
        normalize_answer(answer) in normalized_prediction
        for answer in answers["text"]
        if normalize_answer(answer)
    )

squad_metric = evaluate.load("squad")
predictions = []
references = []

total_time = 0.0

print("Evaluating Optimized RAG...")

eval_dataset = load_dataset("squad", split="validation")
eval_subset = eval_dataset.select(range(0, 200))

for i, example in enumerate(tqdm(eval_subset)):
    start_time = time.time()

    output = rag_chain.invoke(example["question"])

    total_time += time.time() - start_time

    predictions.append({
        "prediction_text": clean_prediction(output),
        "id": example["id"]
    })

    references.append({
        "answers": example["answers"],
        "id": example["id"]
    })

# Sanity check a few examples before trusting the aggregate metric.
for pred, ref in list(zip(predictions, references))[:5]:
    print({"prediction": pred["prediction_text"], "gold": ref["answers"]["text"]})

eval_results = squad_metric.compute(
    predictions=predictions,
    references=references
)

contains_accuracy = 100 * sum(
    contains_gold_answer(pred["prediction_text"], ref["answers"])
    for pred, ref in zip(predictions, references)
) / len(predictions)

avg_latency = total_time / len(eval_subset)

print(
    f"\nFinal Metrics:\n"
    f"Exact Match: {eval_results['exact_match']}\n"
    f"Accuracy: {eval_results['exact_match']}\n"
    f"Contains Accuracy: {contains_accuracy}\n"
    f"F1: {eval_results['f1']}\n"
    f"Average Latency: {avg_latency:.4f} seconds/example"
)

Using Python 3.12.13 environment at: /usr
Resolved 45 packages in 165ms
Prepared 1 package in 27ms
Installed 1 package in 4ms
 + evaluate==0.4.6


Evaluating Optimized RAG...


100%|██████████| 200/200 [08:17<00:00,  2.49s/it]

{'prediction': 'The Denver Broncos', 'gold': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']}
{'prediction': 'The Carolina Panthers', 'gold': ['Carolina Panthers', 'Carolina Panthers', 'Carolina Panthers']}
{'prediction': "Super Bowl 50 took place at Levi's Stadium in Santa Clara, California.", 'gold': ['Santa Clara, California', "Levi's Stadium", "Levi's Stadium in the San Francisco Bay Area at Santa Clara, California."]}
{'prediction': 'The Denver Broncos defeated the Carolina Panthers 24-10 in Super Bowl 50.', 'gold': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos']}
{'prediction': "The 50th Super Bowl was played on February 7, 2016, at Levi's Stadium in Santa Clara,", 'gold': ['gold', 'gold', 'gold']}

Final Metrics:
Exact Match: 28.5
Accuracy: 28.5
Contains Accuracy: 45.5
F1: 37.98329860937885
Average Latency: 2.4874 seconds/example
